# 03 · A máquina que entende **postura** — pose, braços e contagem de movimento

A demo mais forte do bloco, porque a plateia participa: você pede para a sala
levantar o braço e o número muda na tela.

O que ela mostra:
1. o **esqueleto** de cada pessoa (17 pontos)
2. **quantas pessoas** estão na cena
3. **quantas com o braço para cima** e quantas para baixo
4. cada ciclo levantar→baixar contado como **uma movimentação**

> Fala de palco: *"ninguém programou 'braço'. O modelo devolve pontos; a regra
> de negócio — o que conta como levantado — sou eu que escrevo. É essa a divisão
> de trabalho entre a IA e você."*

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
DRIVE = "/content/drive/MyDrive/PALESTRA-IA"
print("palco configurado · raiz no Drive:", DRIVE)

In [ ]:
pose = YOLO(f"{DRIVE}/00-pesos/yolo11n-pose.pt")
print("modelo de pose pronto")

In [ ]:
# ── a regra: o que e "braco levantado" ──
#
# O modelo devolve 17 pontos por pessoa (padrao COCO):
#   5 ombro esq   6 ombro dir   7 cotovelo esq  8 cotovelo dir
#   9 pulso esq  10 pulso dir
#
# Em imagem o eixo Y cresce PARA BAIXO: pulso acima do ombro = y MENOR.
#
# ARMADILHA REAL (custou uma demo quebrada em teste): ponto que o modelo nao
# enxerga volta como (0, 0) com confianca baixa — e (0,0) fica no TOPO da
# imagem, ou seja, um pulso escondido seria lido como "braco levantado".
# Por isso todo ponto passa por um piso de confianca antes de valer.

OMBRO_E, OMBRO_D, PULSO_E, PULSO_D = 5, 6, 9, 10
CONF_MIN = 0.5          # abaixo disso o ponto nao existe para nos

def braco_levantado(pontos, confs):
    """True se QUALQUER pulso visivel estiver acima do ombro do mesmo lado."""
    for ombro, pulso in ((OMBRO_E, PULSO_E), (OMBRO_D, PULSO_D)):
        if confs[ombro] < CONF_MIN or confs[pulso] < CONF_MIN:
            continue                      # ponto nao confiavel: ignora o lado
        if pontos[pulso][1] < pontos[ombro][1]:
            return True
    return False

### Numa imagem parada (o aquecimento)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
entradas = [e for e in sorted(glob.glob(f"{DRIVE}/03-pose/entrada/*"))
            if e.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
IMG = entradas[0] if entradas else "https://ultralytics.com/images/bus.jpg"

r = pose.predict(IMG, conf=.35, verbose=False)[0]
im = r.plot(line_width=4, kpt_radius=8)

pessoas = len(r.boxes)
cima = 0
if r.keypoints is not None and pessoas:
    xy = r.keypoints.xy.cpu().numpy()
    cf = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else None
    if cf is not None:
        cima = sum(braco_levantado(p, c) for p, c in zip(xy, cf))

print(f"pessoas: {pessoas}   braco para cima: {cima}   para baixo: {pessoas - cima}")
plt.figure(); plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"{pessoas} pessoas · {cima} com o braço para cima"); plt.tight_layout(); plt.show()

### O painel ao vivo

Roda sobre um vídeo da pasta `03-pose/entrada/`. O `track` mantém **o mesmo
número** em cima de cada pessoa entre os quadros — é isso que permite contar
*movimentos* em vez de só contar braços.

In [ ]:
# ── processa o video, conta movimentos e grava o resultado anotado ──
import cv2, glob, numpy as np
from collections import defaultdict

videos = [v for v in sorted(glob.glob(f"{DRIVE}/03-pose/entrada/*"))
          if v.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))]
if not videos:
    print("Nenhum video em 03-pose/entrada/ — pule para a celula da webcam.")
else:
    VIDEO = videos[0]
    print("processando:", VIDEO)

    estado = {}                    # id -> braco estava levantado?
    movimentos = defaultdict(int)    # id -> ciclos completos
    serie = []                       # (frame, pessoas, cima) para o grafico

    cap = cv2.VideoCapture(VIDEO)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    saida = f"{DRIVE}/03-pose/saida/pose_anotado.mp4"
    vw = cv2.VideoWriter(saida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    n = 0
    for res in pose.track(VIDEO, stream=True, persist=True, conf=.35, verbose=False):
        n += 1
        frame = res.plot(line_width=3, kpt_radius=6)
        pessoas = len(res.boxes) if res.boxes is not None else 0
        cima = 0

        if res.keypoints is not None and pessoas and res.boxes.id is not None:
            ids = res.boxes.id.cpu().numpy().astype(int)
            xy = res.keypoints.xy.cpu().numpy()
            cf = res.keypoints.conf.cpu().numpy() if res.keypoints.conf is not None else None
            if cf is not None:
                for pid, p, c in zip(ids, xy, cf):
                    lev = braco_levantado(p, c)
                    cima += lev
                    # ciclo completo = subiu e depois desceu
                    if estado.get(pid) and not lev:
                        movimentos[pid] += 1
                    estado[pid] = lev

        serie.append((n, pessoas, cima))

        # painel desenhado no proprio quadro, em tamanho de palco
        total_mov = sum(movimentos.values())
        cv2.rectangle(frame, (0, 0), (w, 92), (13, 17, 23), -1)
        cv2.putText(frame, f"PESSOAS {pessoas}", (24, 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (230, 237, 243), 4)
        cv2.putText(frame, f"BRACO CIMA {cima}", (int(w*.34), 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (168, 224, 63), 4)
        cv2.putText(frame, f"MOVIMENTOS {total_mov}", (int(w*.68), 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (92, 92, 255), 4)
        vw.write(frame)

    cap.release(); vw.release()
    print(f"{n} quadros · movimentos contados: {sum(movimentos.values())}")
    print("salvo em", saida)

In [ ]:
# ── a serie temporal: como a sala reagiu, quadro a quadro ──
import matplotlib.pyplot as plt
if "serie" in dir() and serie:
    f = [s[0] for s in serie]; p = [s[1] for s in serie]; c = [s[2] for s in serie]
    fig, ax = plt.subplots()
    ax.plot(f, p, lw=4, color=CINZA, label="pessoas na cena")
    ax.plot(f, c, lw=5, color=VERDE, label="com o braço para cima")
    ax.fill_between(f, c, color=VERDE, alpha=.18)
    ax.set_xlabel("quadro"); ax.set_ylabel("quantidade")
    ax.set_title("A sala, medida quadro a quadro")
    ax.legend(loc="upper left")
    plt.tight_layout(); plt.show()
else:
    print("rode a celula anterior com um video primeiro")

### Webcam ao vivo (opcional — leia antes de usar no palco)

O Colab roda num servidor remoto: a webcam é capturada **no seu navegador** e
cada quadro sobe para o servidor. Resultado: funciona, mas gira a **poucos
quadros por segundo**, com atraso perceptível. Para a plateia isso lê como
"travado", não como "ao vivo".

**Recomendação:** use esta célula só para uma foto única — o efeito de ver a
própria sala analisada é imediato e não depende de fluidez. Para vídeo fluido,
rode o mesmo código na sua máquina, fora do Colab.

In [ ]:
# ── captura UM quadro da webcam e analisa ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2, numpy as np, matplotlib.pyplot as plt

def foto(nome="webcam.jpg", qualidade=0.92):
    display(Javascript("""
      async function tirar(qualidade) {
        const div = document.createElement('div');
        const capturar = document.createElement('button');
        capturar.textContent = 'CLIQUE PARA CAPTURAR';
        capturar.style.cssText = 'font-size:22px;padding:14px 28px;margin:10px;cursor:pointer';
        div.appendChild(capturar);
        const video = document.createElement('video');
        video.style.display = 'block'; video.style.maxWidth = '100%';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div); div.appendChild(video);
        video.srcObject = stream; await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((r) => capturar.onclick = r);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop(); div.remove();
        return canvas.toDataURL('image/jpeg', qualidade);
      }
    """))
    dados = eval_js(f"tirar({qualidade})")
    binario = b64decode(dados.split(',')[1])
    with open(nome, 'wb') as f:
        f.write(binario)
    return nome

arq = foto()
r = pose.predict(arq, conf=.35, verbose=False)[0]
pessoas = len(r.boxes); cima = 0
if r.keypoints is not None and pessoas and r.keypoints.conf is not None:
    xy = r.keypoints.xy.cpu().numpy(); cf = r.keypoints.conf.cpu().numpy()
    cima = sum(braco_levantado(p, c) for p, c in zip(xy, cf))

plt.figure(); plt.imshow(cv2.cvtColor(r.plot(line_width=4, kpt_radius=8), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title(f"{pessoas} pessoas · {cima} com o braço levantado")
plt.tight_layout(); plt.show()